# Практика · NER

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє: [homework.html](homework.html)

> ⏱ Зошит навчає **три лінійні моделі** на сотні тисяч токенів кожну.
> Заміряно: **близько двох хвилин процесорного часу** на чотирьох ядрах без
> відеокарти, в **один потік**. Стінного часу буде більше й залежить від того,
> чим ще зайнята машина: на завантаженій у нас виходило близько чотирьох хвилин.
> Останньою клітинкою зошит друкує власний процесорний час — звір із цим числом.

## Що ми тут робимо

Задача: знайти в тексті **іменовані сутності** — назви програм, організацій і
ліцензій — і сказати, що з них чим є. Ми пройдемо весь шлях: зберемо дані,
зробимо розмітку, навчимо модель і **зміряємо її пʼятьма різними способами**.

Головне питання зошита не «яка модель краща», а **«чи міряє наше число те, що
ми думаємо»**. Тому порядок дій такий:

1. збираємо корпус із описів програмних пакетів, що стоять на цій машині;
2. робимо розмітку **автоматично**, проєкцією списків назв, — і одразу
   **дивимось на неї очима**, бо це найдешевша перевірка на світі;
3. рахуємо **точність по токенах** і рубіж «відповідати <code>O</code> на все»;
4. пишемо **власний F1 по сутностях** і звіряємо його з бібліотечним;
5. ділимо сутності на **бачені** й **небачені** — і бачимо, звідки береться
   якість;
6. будуємо рубіж «**самий словник, без навчання**» — і дивимось, хто виграє;
7. окремо розбираємо **українську**: чому відмінювання ламає пошук за списком.

⚠️ **Розмітка тут срібна, а не еталонна.** Її ніхто не робив руками: ми беремо
списки назв із типізованих полів бази пакетів і механічно шукаємо ці рядки в
текстах. Мітки написала людина — але в **іншому полі** й для іншої мети. Усе,
що зошит порахує, обмежене якістю цієї розмітки, і один із розділів саме про
те, наскільки вона погана.

⚠️ **Твої числа не збігатимуться з нашими.** Корпус береться з бази
встановлених у тебе пакетів, а вона в кожного своя. Відтворюється **форма**
результату — співвідношення між метриками, — а не самі підрахунки. Зошит
друкує свої числа; порівнюй їх із лекційними за напрямком, а не за значенням.

## 0 · Середовище: чому ми фіксуємо потоки

Зошит міряє час, а час на спільній машині бреше двома способами.

**Стінний годинник** показує, скільки минуло реального часу. Якщо поряд
рахується щось іще, він покаже більше, і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує лише той час, коли
процесор працював над нашою програмою. Але й у нього є пастка: якщо бібліотека
лінійної алгебри розкладає роботу на потоки, то потоки, які **чекають**, теж
зараховуються як робота. Тому спершу фіксуємо один потік — і робимо це **до**
імпорту `numpy`, бо змінні середовища читаються під час завантаження
бібліотеки.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту numpy: інакше бібліотеки вже
# запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import re, sys, time, random, subprocess, glob, gettext
from collections import Counter, defaultdict
import numpy as np

STARTED = time.process_time()          # звідси рахуємо власний час зошита

print('python              ', sys.version.split()[0])
print('numpy               ', np.__version__)
print('ядер у машині       ', os.cpu_count())
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · Дані: описи програмних пакетів

Курс не тримає текстів у репозиторії. Замість цього ми беремо те, що вже лежить
на машині читача: **описи встановлених пакетів**. Це англійська технічна проза,
писана людьми, і в ній густо назв програм — саме те, що потрібно для NER.

Дістаємо їх командою `rpm`. У кожного запису беремо пʼять полів:

| поле | навіщо |
|---|---|
| `NAME` | назва пакета — з неї збиратимемо список продуктів |
| `SUMMARY` | коротке резюме, тут не використовуємо |
| `LICENSE` | ідентифікатор ліцензії |
| `URL` | адреса сайту проєкту |
| `DESCRIPTION` | сам текст, у якому шукатимемо сутності |

⚠️ **Це працює на системах із `rpm`** (Fedora, RHEL, openSUSE). На Debian і
Ubuntu ту саму інформацію дає `dpkg-query -W -f`, і нижче є така гілка —
але **ми її не перевіряли**, бо Debian у нас немає. Якщо вона в тебе не
спрацює, це наша недоробка, а не твоя.

Якщо жодна команда не працює, зошит скаже це людською мовою й зупиниться —
без стека на двадцять рядків.

In [ ]:
FIELDS = ('NAME', 'SUMMARY', 'LICENSE', 'URL', 'DESCRIPTION')

def load_packages(min_words=20):
    """Список словників про пакети з описом щонайменше на min_words слів.

    Розділювачі беремо службові (\x1f між полями, \x1e між записами), бо
    в описах трапляються і коми, і табуляції, і переводи рядка.
    """
    query = '\x1f'.join('%{' + f + '}' for f in FIELDS) + '\x1e'
    raw = ''
    try:                                   # шлях rpm — перевірений
        raw = subprocess.run(['rpm', '-qa', '--qf', query],
                             capture_output=True, text=True, timeout=120).stdout
    except (FileNotFoundError, subprocess.SubprocessError):
        pass
    if not raw.strip():                    # шлях dpkg — НЕ перевірений, див. текст вище
        deb = '\x1f'.join(['${Package}', '${binary:Summary}', 'невідомо',
                           '${Homepage}', '${Description}']) + '\x1e'
        try:
            raw = subprocess.run(['dpkg-query', '-W', '-f', deb],
                                 capture_output=True, text=True, timeout=120).stdout
        except (FileNotFoundError, subprocess.SubprocessError):
            raw = ''
    records = []
    for chunk in raw.split('\x1e'):
        parts = chunk.split('\x1f')
        if len(parts) != len(FIELDS):
            continue
        rec = dict(zip([f.lower() for f in FIELDS], [p.strip() for p in parts]))
        rec['desc'] = rec.pop('description')
        if rec['desc'] and rec['desc'] != '(none)' and len(rec['desc'].split()) >= min_words:
            records.append(rec)
    records.sort(key=lambda r: r['name'])   # порядок однаковий у всіх запусків
    return records


packages = load_packages()

if len(packages) < 200:
    print('Не вдалося зібрати корпус описів пакетів.')
    print()
    print('Зошит бере тексти з бази встановлених програм. Він працює там, де є')
    print('команда rpm (Fedora, RHEL, openSUSE) або dpkg-query (Debian, Ubuntu).')
    print('Знайдено записів з описом на 20+ слів:', len(packages), '- цього замало.')
    print()
    print('Що можна зробити: запустити зошит на машині з Linux, де встановлено')
    print('хоча б кількасот пакетів. Усередині контейнера з мінімальним образом')
    print('описів зазвичай немає.')
    raise SystemExit('немає даних - далі рахувати нема чого')

n_words = sum(len(r['desc'].split()) for r in packages)
print('пакетів з описом на 20+ слів:', len(packages))
print('слів в описах разом         :', n_words)
print('слів в описі: медіана       :', int(np.median([len(r['desc'].split()) for r in packages])))
print()
print('приклад запису:')
example = packages[len(packages) // 2]
for k in ('name', 'license', 'url'):
    print('  %-8s %s' % (k, example[k][:70]))
print('  desc    ', example['desc'][:150].replace('\n', ' '), '...')

## 2 · Розмітка: звідки візьмемо мітки

Розміченого корпусу NER у нас немає, і зробити його руками — це тижні роботи.
Тому робимо **срібну розмітку**: збираємо три списки назв із **типізованих
полів** тієї самої бази й механічно шукаємо ці рядки в описах.

| список | звідки | тип сутності |
|---|---|---|
| назви програм | поле `NAME` усіх пакетів | `PROD` |
| ідентифікатори ліцензій | поле `LICENSE`, розбите по «and»/«or»/«with» | `LIC` |
| організації | **реєстрована частина** хоста з поля `URL` | `ORG` |

Ключове: **мітки не вигадані**. Кожен рядок у списку написала людина —
супровідник пакета. Але жодна людина не читала опису й не казала, що саме
в ньому є сутністю. Саме тому розмітка срібна, а не еталонна.

⚠️ Зверни увагу на слово «реєстрована» в третьому рядку таблиці. Тут захована
історія, до якої ми повернемось у розділі 5, і вона коштувала цьому корпусу
майже третини всіх міток. Поки що просто зафіксуй: з адреси
`fonts.google.com` ми беремо `google`, а не `fonts`.

Дуже короткі рядки викидаємо: односимвольні «назви» дали б шум. І викидаємо
кілька службових англійських слів, які випадково збігаються з іменами пакетів
(`make`, `less`, `file`, `time`…) — інакше в списку продуктів опиниться
половина звичайної лексики. Побачимо далі, що цього фільтра **дуже мало**.

In [ ]:
STOPNAMES = {'which', 'make', 'less', 'file', 'time', 'info', 'more', 'base',
             'text', 'data', 'tools', 'library', 'system', 'open', 'source',
             'common', 'util'}

# у зонах на кшталт .co.uk або .org.uk другий рівень — не імʼя організації
SECOND_LEVEL = {'co', 'com', 'org', 'net', 'ac', 'gov', 'edu'}

def registrable_part(url):
    """gnu.org -> gnu · fonts.google.com -> google · plugin.org.uk -> plugin"""
    m = re.match(r'https?://(?:www\.)?([^/:]+)', url or '')
    if not m:
        return ''
    parts = [p for p in m.group(1).split('.') if p]
    if len(parts) < 2:
        return ''
    head = parts[-2]                       # передостання частина імені хоста
    if head in SECOND_LEVEL and len(parts) >= 3:
        head = parts[-3]                   # .co.uk і подібні: беремо на крок лівіше
    return head


def build_gazetteer(records, org_rule=registrable_part):
    """Три списки назв, зібрані з типізованих полів бази пакетів.

    org_rule — окремим аргументом навмисно: у розділі 5 ми підставимо сюди
    старе правило й порівняємо, скільки міток воно давало.
    """
    all_names = {r['name'] for r in records}
    products = {n for n in all_names if len(n) > 3}
    # урізаний префікс (open-vm-tools -> open) беремо, лише якщо він сам є пакетом
    products |= {r['name'].split('-')[0] for r in records} & all_names
    products = {p for p in products if p.lower() not in STOPNAMES}

    licenses = set()
    for r in records:
        # «BSD and Python and Unicode» -> три окремі рядки
        for piece in re.split(r'\s+(?:AND|OR|and|or|WITH)\s+|[()]', r['license'] or ''):
            piece = piece.strip()
            if len(piece) > 3:
                licenses.add(piece)

    orgs = set()
    for r in records:
        head = org_rule(r['url'])
        if len(head) > 3:
            orgs.add(head)
    return {'PROD': products, 'LIC': licenses, 'ORG': orgs}


gazetteer = build_gazetteer(packages)
for kind in ('PROD', 'ORG', 'LIC'):
    print('%-5s %5d рядків   приклади: %s'
          % (kind, len(gazetteer[kind]), ', '.join(sorted(gazetteer[kind])[:5])))

## 3 · Проєкція списків у BIO

Тепер треба перетворити «знайдені в тексті рядки» на мітки для кожного токена.
Схема **BIO**: `B-ТИП` — перший токен сутності, `I-ТИП` — її продовження,
`O` — токен поза сутностями.

Три рішення тут не косметичні.

**Перше: довший збіг виграє в коротшого.** Якщо в тексті стоїть
`GPL-2.0-or-later`, а в списку є і `GPL-2.0-or-later`, і `GPL`, ми беремо
довший. Інакше сутність розірветься.

**Друге: токенізатор мусить різати дефіси й крапки.** Спокуса розбити текст
пробілами велика, але тоді `GPL-2.0-or-later` стане **одним** токеном, усі
сутності виявляться однослівними, і в розмітці не буде жодної мітки `I-` —
тобто вся структура BIO зникне й міряти межі стане нічим.

**Третє: коли той самий проміжок знайшли два списки, комусь треба віддати
перевагу.** Рядок `curl` законно є і назвою програми, і назвою ліцензії, під
якою вона випущена. У тексті опису це майже завжди продукт, тому беремо
`PROD`, потім `ORG`, потім `LIC`. Це **рішення, а не істина**, і в розділі 5
ми побачимо, чого воно не лікує.

Межі збігу перевіряємо переглядом назад і вперед: рядок `perl` не повинен
матчитись усередині `perl-Encode`.

In [ ]:
def gazetteer_pattern(strings):
    """Один регулярний вираз на весь список. Довші рядки стоять раніше,
    тому чергування вибирає найдовший збіг."""
    ordered = sorted(strings, key=len, reverse=True)
    return re.compile(r'(?<![\w-])(' + '|'.join(re.escape(s) for s in ordered) + r')(?![\w-])')


patterns = {kind: gazetteer_pattern(strings) for kind, strings in gazetteer.items()}

# ⚠️ дефіс і крапка — окремі токени, інакше BIO лишиться без структури
TOKEN = re.compile(r'[A-Za-z]+|[0-9]+|[^\sA-Za-z0-9]')
# за однакового проміжку продукт виграє в організації, організація — у ліцензії
TYPE_PRIORITY = {'PROD': 0, 'ORG': 1, 'LIC': 2}

def tag_text(text, patterns):
    """(токени, мітки BIO) для одного опису."""
    found = []
    for kind, pattern in patterns.items():
        for m in pattern.finditer(text):
            found.append((m.start(), m.end(), kind))
    # раніший початок першим; за однакового початку — довший збіг; за однакової
    # довжини — тип із меншим номером пріоритету
    found.sort(key=lambda s: (s[0], -(s[1] - s[0]), TYPE_PRIORITY.get(s[2], 9)))
    kept, last_end = [], -1
    for start, end, kind in found:
        if start >= last_end:
            kept.append((start, end, kind))
            last_end = end

    tokens, labels = [], []
    for m in TOKEN.finditer(text):
        a, b = m.span()
        label = 'O'
        for start, end, kind in kept:
            if a >= start and b <= end:
                label = ('B-' if a == start else 'I-') + kind
                break
        tokens.append(m.group())
        labels.append(label)
    return tokens, labels


documents = []
for r in packages:
    toks, labs = tag_text(r['desc'], patterns)
    if toks:
        documents.append((r['name'], toks, labs))

print('розмічених описів:', len(documents))
print()
# показуємо перший опис, у якому є хоча б дві сутності різних типів
for name, toks, labs in documents:
    kinds = {x[2:] for x in labs if x != 'O'}
    if len(kinds) >= 2 and len(toks) < 60:
        print('пакет', name)
        for t, l in zip(toks, labs):
            print('   %-18s %s' % (t, l if l != 'O' else '·'))
        break

## 4 · Що собою являє розмічений корпус

Два числа тут задають усе, що буде далі. Порахуймо їх окремо і вголос.

In [ ]:
def spans_of(labels):
    """Список сутностей (початок, кінець_невключно, тип) із рядка міток BIO.

    Читаємо мʼяко: якщо I- зустрівся без попереднього B-, він усе одно
    відкриває сутність. Так само поводиться бібліотека seqeval у типовому
    режимі, і нижче ми це перевіримо.
    """
    out, current = [], None
    for i, x in enumerate(labels):
        if x.startswith('B-'):
            if current:
                out.append(tuple(current))
            current = [i, i + 1, x[2:]]
        elif x.startswith('I-'):
            if current and current[2] == x[2:]:
                current[1] = i + 1
            else:
                if current:
                    out.append(tuple(current))
                current = [i, i + 1, x[2:]]
        else:
            if current:
                out.append(tuple(current))
                current = None
    if current:
        out.append(tuple(current))
    return out


n_tokens = sum(len(t) for _, t, _ in documents)
n_outside = sum(1 for _, _, l in documents for x in l if x == 'O')
lengths = Counter()
by_type = Counter()
for _, toks, labs in documents:
    for a, b, kind in spans_of(labs):
        lengths[b - a] += 1
        by_type[kind] += 1
n_entities = sum(lengths.values())

print('токенів у корпусі         :', n_tokens)
print('сутностей                 :', n_entities, dict(by_type))
print()
print('частка токенів з міткою O : %.4f   <- стільки дасть відповідь «усе O»'
      % (n_outside / n_tokens))
print('однослівних сутностей     : %.4f' % (lengths[1] / n_entities))
print()
print('довжина сутності в токенах:')
for k in sorted(lengths):
    print('   %2d : %5d  (%.4f)' % (k, lengths[k], lengths[k] / n_entities))

## 5 · Найдешевша перевірка на світі: подивитись на розмітку очима

Зараз спокусливо піти навчати модель. Не йдімо. Спершу зробімо те, що коштує
одну клітинку й рятує від усього подальшого сорому: **надрукуймо найчастіші
сутності** й прочитаймо їх.

Автоматична перевірка тут не допоможе в принципі. Розмітка внутрішньо
несуперечлива: список склався, збіги знайшлися, мітки поставилися. Помилка в
ній систематична, тому вона однакова і в навчальній, і в перевірній частині —
і жодна метрика на неї не поскаржиться.

In [ ]:
surface = Counter()
surface_type = defaultdict(Counter)
for _, toks, labs in documents:
    for a, b, kind in spans_of(labs):
        text = ' '.join(toks[a:b])
        surface[text] += 1
        surface_type[text][kind] += 1

print('різних поверхневих форм:', len(surface), 'на', n_entities, 'уживань')
print()
print('%-16s %-6s %6s' % ('форма', 'тип', 'разів'))
for text, count in surface.most_common(15):
    print('%-16s %-6s %6d' % (text, surface_type[text].most_common(1)[0][0], count))

Читаємо цей список. Якщо в ньому нагорі стоять слова на кшталт **plugin**,
**free**, **archive**, **case** — це не організації, а звичайні англійські
іменники. Зараз ми дізнаємось, звідки вони взялись, і заразом побачимо, як
виглядала та сама розмітка **до** одного виправлення.

Наступна клітинка бере кожну підозрілу форму й **простежує її до джерела**:
який запис у базі поклав цей рядок у список.

In [ ]:
def trace_org(word, org_rule=registrable_part):
    """Які записи бази поклали цей рядок у список організацій."""
    return [(r['name'], r['url']) for r in packages if org_rule(r['url']) == word]


def trace_lic(word):
    """Які записи бази поклали цей рядок у список ліцензій."""
    out = []
    for r in packages:
        pieces = [p.strip() for p in re.split(r'\s+(?:AND|OR|and|or|WITH)\s+|[()]',
                                              r['license'] or '')]
        if word in pieces:
            out.append((r['name'], r['license']))
    return out


print('%-12s %6s  %-6s  %s' % ('форма', 'міток', 'список', 'скільки записів її породили'))
for text, count in surface.most_common(20):
    kind = surface_type[text].most_common(1)[0][0]
    if kind == 'ORG':
        src = trace_org(text)
    elif kind == 'LIC':
        src = trace_lic(text)
    else:
        src = [(r['name'], '') for r in packages if r['name'] == text]
    if src and len(src) <= 4:            # мало джерел, багато міток — підозріло
        print('%-12s %6d  %-6s  %d: %s'
              % (text, count, kind, len(src),
                 ', '.join('%s (%s)' % (n, h[:34]) for n, h in src[:2])))

Ось де ламається автоматична розмітка, і ламається вона **множником**: один
запис кладе в список звичайне слово, і кожне вживання цього слова в
**будь-якому** описі стає сутністю.

А тепер найцікавіше. Правило, яким ми беремо організацію з адреси, у цьому
зошиті вже **виправлене**: воно бере реєстровану частину хоста. Спочатку
воно брало **першу** частину — і тоді `fonts.google.com` давало не `google`,
а `fonts`. Подивімось, скільки це коштувало.

In [ ]:
def first_part(url):
    """Старе правило: перша частина імені хоста. fonts.google.com -> fonts"""
    m = re.match(r'https?://(?:www\.)?([^/]+)', url or '')
    return m.group(1).split('.')[0] if m else ''


def tag_all(org_rule):
    """Уся розмітка корпусу за заданим правилом ORG: (сутності за типами,
    токенів у сутностях, описів із сутністю, найчастіші ORG-форми)."""
    g = build_gazetteer(packages, org_rule=org_rule)
    pats = {k: gazetteer_pattern(v) for k, v in g.items()}
    by_kind = Counter()
    org_forms = Counter()
    entity_tokens = with_entity = total_tokens = 0
    for r in packages:
        toks, labs = tag_text(r['desc'], pats)
        total_tokens += len(toks)
        entity_tokens += sum(1 for x in labs if x != 'O')
        if any(x != 'O' for x in labs):
            with_entity += 1
        for a, b, kind in spans_of(labs):
            by_kind[kind] += 1
            if kind == 'ORG':
                org_forms[' '.join(toks[a:b])] += 1
    return {'gaz': g, 'by_kind': by_kind, 'org_forms': org_forms,
            'entity_tokens': entity_tokens, 'with_entity': with_entity,
            'total_tokens': total_tokens}


old = tag_all(first_part)
new = tag_all(registrable_part)

print('%-28s %14s %14s' % ('', 'перша частина', 'реєстрована'))
print('%-28s %14d %14d' % ('рядків у списку ORG', len(old['gaz']['ORG']), len(new['gaz']['ORG'])))
for kind in ('PROD', 'ORG', 'LIC'):
    print('%-28s %14d %14d' % ('сутностей ' + kind, old['by_kind'][kind], new['by_kind'][kind]))
o_all, n_all = sum(old['by_kind'].values()), sum(new['by_kind'].values())
print('%-28s %14d %14d' % ('усього сутностей', o_all, n_all))
print('%-28s %14d %14d' % ('токенів у сутностях', old['entity_tokens'], new['entity_tokens']))
print('%-28s %14.4f %14.4f' % ('частка токенів O',
                               1 - old['entity_tokens'] / old['total_tokens'],
                               1 - new['entity_tokens'] / new['total_tokens']))
print('%-28s %14d %14d' % ('описів із сутністю', old['with_entity'], new['with_entity']))
print()
print('зникло міток: %d, тобто %.4f старої розмітки' % (o_all - n_all, (o_all - n_all) / o_all))

Майже третина міток була сміттям — і породило її кілька записів із 1733.

Тепер найважливіше питання цього розділу, і воно не про те, добре чи погано
ми полагодили. Питання таке: **чи зникла вада взагалі?** Пошукаймо ту саму
ознаку — рядок у списку `ORG`, який поклав один-єдиний запис, а міток дав
багато — у **виправленій** розмітці.

In [ ]:
def single_source_org(state, org_rule, min_labels=20):
    """ORG-рядки, які поклав один-два записи, а міток вони дали багато."""
    rows = []
    for word, count in state['org_forms'].items():
        sources = [r['name'] for r in packages if org_rule(r['url']) == word]
        if sources and len(sources) <= 2 and count >= min_labels:
            urls = [r['url'] for r in packages if org_rule(r['url']) == word]
            rows.append((count, word, len(sources), sources[0], urls[0]))
    rows.sort(reverse=True)
    return rows


for title, state, rule in (('СТАРЕ правило', old, first_part),
                           ('НОВЕ правило', new, registrable_part)):
    rows = single_source_org(state, rule)
    total_labels = sum(state['by_kind'].values())
    noisy = sum(r[0] for r in rows)
    print(title)
    for count, word, n_src, who, url in rows:
        print('   %-10s %4d міток  від %d запису (%s, %s)'
              % (word, count, n_src, who, url[:40]))
    print('   разом: %d міток = %.4f усієї розмітки' % (noisy, noisy / total_labels))
    print()

Ось головний висновок розділу, і він неприємніший за «ми знайшли й полагодили».

Виправлення зменшило біду в кілька разів — і **не прибрало її**. У новій
розмітці й далі є звичайні англійські слова, які потрапили в список
організацій через адресу одного пакета.

Подивись на рядок `case`. Він походить із `tiswww.case.edu` — сайту
Університету Кейс Вестерн Резерв. Виправлене правило витягло звідти `case`
**абсолютно правильно**: це справді реєстрована частина домену і справді
назва організації. Хибних міток однаково два десятки — бо «case» ще й
звичайне англійське слово, а газетир не бачить речення.

**Газетир, зібраний автоматично, не можна полагодити — його можна лише
зробити менш хибним.** Кожне виправлення прибирає той різновид помилки, який
ти помітив, і лишає ті, яких не помітив.

І є ще один різновид, який не виправити взагалі, бо це не помилка, а
властивість даних: той самий рядок законно належить двом типам.

In [ ]:
# рядки, що є водночас іменами пакетів і назвами ліцензій
both = sorted(gazetteer['LIC'] & gazetteer['PROD'])
print('рядків одразу в списках LIC і PROD:', len(both), '->', both)
print('(їх розводить правило пріоритету: PROD виграє)')
print()

lic_forms = Counter()
for _, toks, labs in documents:
    for a, b, kind in spans_of(labs):
        if kind == 'LIC':
            lic_forms[' '.join(toks[a:b])] += 1
total_lic = sum(lic_forms.values())
print('найчастіші сутності типу LIC:')
for word, count in lic_forms.most_common(6):
    print('   %-14s %4d' % (word, count))
top2 = sum(c for _, c in lic_forms.most_common(2))
print()
print('дві найчастіші форми дають %d міток із %d = %.4f усього типу LIC'
      % (top2, total_lic, top2 / total_lic))
print()
print('⚠️ Подивись, які це форми. Якщо це Python і Unicode — то вони стоять у полі')
print('   LICENSE («BSD and Python and Unicode»), але в тексті опису це назви')
print('   продуктів, а не ліцензій. Правило пріоритету їх НЕ рятує: імені пакета')
print('   «Python» у базі немає (є python3, python3-idna), тож змагатись нема з чим.')
print('   Тобто високе число типу LIC далі означатиме почасти не «модель упізнає')
print('   ліцензії», а «модель вивчила, що рядок Python тут має мітку LIC».')

## 6 · Поділ: за пакетом, не за реченнями

Ділити треба **за документом**, а не випадковими реченнями. Якщо речення
одного опису потраплять і в навчальну, і в перевірну частину, та сама назва
опиниться по обидва боки — і перевірна вибірка міряла б памʼять, а не вміння.

Беремо 80 % пакетів у навчання, 20 % на перевірку. Порядок пакетів у базі
алфавітний, тому перемішуємо з фіксованим зерном: «перші 20 %» були б не
меншою вибіркою, а вужчим доменом (усі назви на `a`).

In [ ]:
def split_by_package(docs, seed):
    """(навчальна, перевірна) — цілими пакетами."""
    names = sorted({d[0] for d in docs})
    random.Random(seed).shuffle(names)          # зерно фіксоване -> поділ відтворюваний
    cut = int(0.8 * len(names))
    train_names = set(names[:cut])
    train = [d for d in docs if d[0] in train_names]
    test = [d for d in docs if d[0] not in train_names]
    return train, test


train_docs, test_docs = split_by_package(documents, 0)
print('навчальних описів:', len(train_docs))
print('перевірних описів:', len(test_docs))
print('токенів у перевірній частині:', sum(len(t) for _, t, _ in test_docs))

## 7 · Ознаки: що модель бачить у кожному токені

Модель — звичайна логістична регресія, яка класифікує **кожен токен окремо**.
Щоб вона могла врахувати контекст, ознаки описують не лише сам токен, а й двох
сусідів ліворуч і двох праворуч.

| ознака | навіщо |
|---|---|
| саме слово | найсильніша ознака: назви повторюються |
| форма слова (`Aaa`, `AAA`, `a0.0`) | назви програм часто мають характерний вигляд |
| префікс і суфікс на три літери | `lib…`, `…utils` — ознака сімейства |
| велика літера, усі великі, є цифра | класика англійського NER |
| довжина, чи це перше слово опису | дешеві структурні підказки |
| ті самі слова й регістр сусідів ±1, ±2 | власне контекст |

Ознак виходить багато й вони текстові, тож переводимо їх у числа
**хешуванням**: кожен рядок ознаки перетворюємо на номер стовпчика функцією
хешування. Це `FeatureHasher`. Плата — рідкісні збіги різних ознак в одному
стовпчику; виграш — не треба зберігати словник ознак.

In [ ]:
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression

def word_shape(word):
    """ImageMagick -> Aaaaaaaaaaa ; GPL-2.0 -> AAA-0.0"""
    return re.sub('[0-9]', '0', re.sub('[A-Z]', 'A', re.sub('[a-z]', 'a', word)))


def features(tokens, i):
    """Словник ознак для токена номер i."""
    word = tokens[i]
    low = word.lower()
    f = {
        'w=' + low: 1,
        'shape=' + word_shape(word): 1,
        'pre3=' + low[:3]: 1,
        'suf3=' + low[-3:]: 1,
        'cap=%s' % word[:1].isupper(): 1,
        'up=%s' % word.isupper(): 1,
        'dig=%s' % any(c.isdigit() for c in word): 1,
        'punct=%s' % bool(re.fullmatch(r'[^\w]+', word)): 1,
        'len=%d' % min(len(word), 12): 1,
        'first=%s' % (i == 0): 1,
    }
    for shift in (-2, -1, 1, 2):
        j = i + shift
        inside = 0 <= j < len(tokens)
        f['w%+d=%s' % (shift, tokens[j].lower() if inside else '<край>')] = 1
        if inside:
            f['cap%+d=%s' % (shift, tokens[j][:1].isupper())] = 1
    return f


# 2**18 стовпчиків: перевірено, що 2**20 дає ті самі числа вчетверо довше
hasher = FeatureHasher(n_features=2 ** 18, input_type='dict')

def vectorize(docs):
    return hasher.transform(features(t, i) for _, t, _ in docs for i in range(len(t)))

def flat_labels(docs):
    return [x for _, _, labs in docs for x in labs]


t0 = time.process_time()
model = LogisticRegression(max_iter=120, C=2.0).fit(vectorize(train_docs),
                                                    flat_labels(train_docs))
print('навчено за %.1f с процесорного часу' % (time.process_time() - t0))
print('класів у моделі:', len(model.classes_), sorted(model.classes_))

## 8 · Перше число: точність по токенах — і рубіж поруч

Найприродніша метрика: скільки токенів дістали правильну мітку. Друкуємо її
**разом із рубежем** — точністю моделі, яка відповідає `O` на все й не вміє
нічого. Окремо перше число не означає нічого.

⚠️ Рубіж рахуємо на **тих самих** перевірних документах, а не по всьому
корпусу. Густота сутностей у перевірній вибірці своя, і порівняння з
загальнокорпусним числом завищило б здобуток моделі.

In [ ]:
def predict_documents(model, docs):
    """Список рядків міток — по одному на документ."""
    flat = list(model.predict(vectorize(docs)))
    out, offset = [], 0
    for _, toks, _ in docs:
        out.append(flat[offset:offset + len(toks)])
        offset += len(toks)
    return out


gold_labels = [labs for _, _, labs in test_docs]
pred_labels = predict_documents(model, test_docs)

flat_gold = [x for row in gold_labels for x in row]
flat_pred = [x for row in pred_labels for x in row]

token_accuracy = sum(a == b for a, b in zip(flat_gold, flat_pred)) / len(flat_gold)
all_o_baseline = sum(1 for x in flat_gold if x == 'O') / len(flat_gold)

print('рубіж «усе O»        : %.4f' % all_o_baseline)
print('точність по токенах  : %.4f' % token_accuracy)
print()
print('здобуто над рубежем  : %.4f пункту' % (token_accuracy - all_o_baseline))
print('було чого здобувати  : %.4f' % (1 - all_o_baseline))
print('використано простору : %.4f' % ((token_accuracy - all_o_baseline) / (1 - all_o_baseline)))

Ось перший урок теми. Число «точність 0.99» виглядає як розвʼязана задача, але
рубіж стоїть майже там само. Уся задача вміщується в кілька сотих, і в цих
сотих модель узяла лише частину.

## 9 · Друге число: F1 по сутностях — і пишемо його самі

Користувачеві потрібен **список сутностей**, а не мітки токенів. Отже, міряти
треба списки: сутність зарахована, тільки якщо збіглися **і початок, і кінець,
і тип**.

Напишемо це самі — тоді видно, що всередині немає магії, — і **звіримо з
бібліотекою** `seqeval`, яку для NER використовують усі.

In [ ]:
def entity_scores(gold_rows, pred_rows):
    """precision, recall, F1 по сутностях. Сутність зарахована лише за
    повного збігу початку, кінця й типу."""
    tp = fp = fn = 0
    for gold, pred in zip(gold_rows, pred_rows):
        gold_set = set(spans_of(gold))
        pred_set = set(spans_of(pred))
        tp += len(gold_set & pred_set)      # знайдено правильно
        fp += len(pred_set - gold_set)      # вигадано зайве
        fn += len(gold_set - pred_set)      # пропущено
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1, tp, fp, fn


our_p, our_r, our_f1, tp, fp, fn = entity_scores(gold_labels, pred_labels)

from seqeval.metrics import f1_score as seqeval_f1, precision_score as seqeval_p
from seqeval.metrics import recall_score as seqeval_r, classification_report

lib_p = seqeval_p(gold_labels, pred_labels)
lib_r = seqeval_r(gold_labels, pred_labels)
lib_f1 = seqeval_f1(gold_labels, pred_labels)

print('%-12s %10s %10s' % ('', 'наше', 'seqeval'))
print('%-12s %10.6f %10.6f' % ('precision', our_p, lib_p))
print('%-12s %10.6f %10.6f' % ('recall', our_r, lib_r))
print('%-12s %10.6f %10.6f' % ('F1', our_f1, lib_f1))

assert np.allclose([our_p, our_r, our_f1], [lib_p, lib_r, lib_f1]), 'розрахунок розійшовся!'
print()
print('✅ збігається')
print()
print('TP =', tp, '· FP =', fp, '· FN =', fn)
print('точність по токенах  : %.4f' % token_accuracy)
print('F1 по сутностях      : %.4f' % our_f1)

Одна модель, один прогін, ті самі виходи — і два числа, які розповідають різні
історії. Перше каже «майже ідеально», друге — «кожна пʼята сутність зіпсована».
Обидва правильні; різниця в тому, **що вони рахують**.

### Строго чи нестрого

`seqeval` уміє рахувати двома способами. **Нестрогий** (типовий) відновлює
сутність навіть із поламаної послідовності: побачивши `O I-PROD`, він усе одно
вважає, що сутність почалась. **Строгий** такий рядок вважає невалідним і
сутності там не бачить. Порахуймо обидва — і заразом порахуймо, скільки
поламаних рядків модель узагалі видає.

In [ ]:
from seqeval.scheme import IOB2

def bio_is_valid(labels):
    """Чи можна прочитати цей рядок як коректний BIO: I-X дозволено лише
    після B-X або I-X того самого типу."""
    previous = 'O'
    for x in labels:
        if x.startswith('I-') and (previous == 'O' or previous[2:] != x[2:]):
            return False
        previous = x
    return True


broken = sum(0 if bio_is_valid(row) else 1 for row in pred_labels)

print('F1 нестрого (типово)      : %.4f' % lib_f1)
print('F1 строго (схема IOB2)    : %.4f'
      % seqeval_f1(gold_labels, pred_labels, mode='strict', scheme=IOB2))
print()
print('передбачень зі зламаним BIO: %d із %d (%.4f)'
      % (broken, len(pred_labels), broken / len(pred_labels)))
print()
print('⚠️ у звіті про систему NER режим треба називати поруч із числом:')
print('   різниця між ними більша, ніж різниця між багатьма моделями')

## 10 · Чому зсунута межа — це дві помилки, а не пів

Порахуймо руками на одному вигаданому реченні (тут синтетика законна: нам
потрібна відома істина, щоб перевірити арифметику метрики).

Еталон має три сутності. Модель угадала всі три типи, але зсунула ліву межу
імені на одне слово. Скільки це помилок?

In [ ]:
demo_tokens = ['Учора', 'Ерік', 'Йонсон', 'підписав', 'угоду',
               'з', 'Nokia', 'у', 'Гельсінкі', '.']

def labels_from_spans(n, entities):
    out = ['O'] * n
    for a, b, kind in entities:
        for j in range(a, b):
            out[j] = ('B-' if j == a else 'I-') + kind
    return out


gold_demo = labels_from_spans(10, [(1, 3, 'PER'), (6, 7, 'ORG'), (8, 9, 'LOC')])
# модель почала імʼя на слово раніше: «Учора Ерік Йонсон»
pred_demo = labels_from_spans(10, [(0, 3, 'PER'), (6, 7, 'ORG'), (8, 9, 'LOC')])

print('%-12s %-8s %-8s' % ('токен', 'еталон', 'модель'))
for t, g, p in zip(demo_tokens, gold_demo, pred_demo):
    print('%-12s %-8s %-8s %s' % (t, g, p, '' if g == p else '<- розбіжність'))

acc_demo = sum(g == p for g, p in zip(gold_demo, pred_demo)) / len(gold_demo)
p_demo, r_demo, f_demo, tp_d, fp_d, fn_d = entity_scores([gold_demo], [pred_demo])

print()
print('точність по токенах : %.4f  (постраждали 2 токени з 10)' % acc_demo)
print('TP = %d · FP = %d · FN = %d' % (tp_d, fp_d, fn_d))
print('F1 по сутностях     : %.4f' % f_demo)
print()
assert fp_d == 1 and fn_d == 1, 'одна зсунута межа мусить дати рівно дві помилки'
print('✅ одна зсунута межа = одна вигадана сутність + одна пропущена')
print('   не пів помилки, а дві — і платять обидві метрики, precision і recall')

## 11 · Три зерна: різниця, менша за розкид, не є різницею

Одне зерно нічого не доводить: інший поділ дасть інші числа. Повторюємо все на
трьох зернах і звітуємо **купу** значень, а не одне середнє.

⚠️ Три зерна — це мінімум для допоміжного заміру. Для головного твердження
курс вимагає пʼять: розкид із трьох — нижня оцінка розкиду, а не розкид.

In [ ]:
SEEDS = [0, 1, 2]
results = []

for seed in SEEDS:
    started = time.process_time()
    tr, te = split_by_package(documents, seed)
    m = LogisticRegression(max_iter=120, C=2.0).fit(vectorize(tr), flat_labels(tr))
    gold_rows = [labs for _, _, labs in te]
    pred_rows = predict_documents(m, te)
    fg = [x for row in gold_rows for x in row]
    fp_ = [x for row in pred_rows for x in row]
    acc = sum(a == b for a, b in zip(fg, fp_)) / len(fg)
    base = sum(1 for x in fg if x == 'O') / len(fg)
    _, _, f1, _, _, _ = entity_scores(gold_rows, pred_rows)
    results.append({'seed': seed, 'base': base, 'acc': acc, 'f1': f1,
                    'train': tr, 'test': te, 'gold': gold_rows, 'pred': pred_rows})
    print('зерно %d: рубіж %.4f · токени %.4f · сутності %.4f   (%.0f с)'
          % (seed, base, acc, f1, time.process_time() - started), flush=True)

def pile(values):
    return '%.4f..%.4f (сер. %.4f)' % (min(values), max(values), sum(values) / len(values))

print()
print('рубіж «усе O»       :', pile([r['base'] for r in results]))
print('точність по токенах :', pile([r['acc'] for r in results]))
print('F1 по сутностях     :', pile([r['f1'] for r in results]))

## 12 · Модель упізнає назви — чи згадує їх?

Головне питання будь-якої системи NER. Перевірка проста: ділимо сутності
перевірної частини надвоє за одним критерієм — чи траплялась така сама
**поверхнева форма** (той самий рядок символів) як сутність у навчанні.

Рахуємо F1 окремо на кожній половині. Важлива деталь: **хибні спрацювання
треба приписати тій половині, до якої вони належать**, а не обом одразу.
Передбачену сутність, якої немає в еталоні, зараховуємо до «бачених», якщо її
рядок був у навчанні, і до «небачених», якщо не був.

In [ ]:
def seen_surfaces(train_docs):
    """Поверхневі форми всіх сутностей навчальної частини."""
    out = set()
    for _, toks, labs in train_docs:
        for a, b, _ in spans_of(labs):
            out.add(' '.join(toks[a:b]))
    return out


def split_scores(train_docs, test_docs, gold_rows, pred_rows):
    """F1 окремо для бачених і небачених поверхневих форм."""
    known = seen_surfaces(train_docs)
    counts = {True: [0, 0, 0], False: [0, 0, 0]}       # [tp, fp, fn]
    for (_, toks, _), gold, pred in zip(test_docs, gold_rows, pred_rows):
        gold_set = set(spans_of(gold))
        pred_set = set(spans_of(pred))
        for span in gold_set:
            bucket = ' '.join(toks[span[0]:span[1]]) in known
            if span in pred_set:
                counts[bucket][0] += 1
            else:
                counts[bucket][2] += 1
        for span in pred_set - gold_set:
            bucket = ' '.join(toks[span[0]:span[1]]) in known
            counts[bucket][1] += 1
    out = {}
    for bucket, (tp, fp, fn) in counts.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        out[bucket] = (p, r, 2 * p * r / (p + r) if p + r else 0.0, tp + fn)
    return out


seen_f1, unseen_f1, share_known = [], [], []
for r in results:
    scores = split_scores(r['train'], r['test'], r['gold'], r['pred'])
    seen_f1.append(scores[True][2])
    unseen_f1.append(scores[False][2])
    share_known.append(scores[True][3] / (scores[True][3] + scores[False][3]))
    print('зерно %d: бачені F1 %.4f (%d сутностей) · небачені F1 %.4f (%d сутностей)'
          % (r['seed'], scores[True][2], scores[True][3],
             scores[False][2], scores[False][3]))

print()
print('F1, форма бачена в навчанні :', pile(seen_f1))
print('F1, форма нова              :', pile(unseen_f1))
print('частка бачених у перевірній :', pile(share_known))
print()
print('розрив у %.2f раза' % (sum(seen_f1) / len(seen_f1) / (sum(unseen_f1) / len(unseen_f1))))

Розрив величезний, і читати його треба буквально: знайому назву модель
впізнає майже завжди, нову — набагато гірше. Це та сама межа, через яку ми
взагалі відмовились від пошуку за словником: модель теж будує собі словник,
просто не список рядків, а ваги при ознаці «саме це слово».

Зверни увагу й на третій рядок: частка бачених форм у перевірній вибірці
велика, тому **загальне F1 стоїть значно ближче до «бачених»**. Змінити цю
частку — і загальне число посунеться, хоч модель та сама. Отже, F1 системи NER
не порівнянне між корпусами, поки не названо, скільки в них нових назв.

## 13 · Рубіж, який не вміє нічого: самий словник

Якщо модель так добре працює зі знайомими рядками — а що буде, якщо викинути
модель і лишити самі знайомі рядки?

Виписуємо з навчальної частини всі поверхневі форми сутностей разом із типами.
Ніякого навчання, ніяких ознак, ніякого контексту. На перевірній частині йдемо
зліва направо й позначаємо кожен збіг зі списком, довший збіг виграє.

In [ ]:
def memory_lookup(train_docs, test_docs, gold_rows):
    """Рубіж «памʼять»: словник із навчальної частини, без навчання."""
    lexicon = defaultdict(Counter)
    for _, toks, labs in train_docs:
        for a, b, kind in spans_of(labs):
            lexicon[' '.join(toks[a:b])][kind] += 1
    lexicon = {text: c.most_common(1)[0][0] for text, c in lexicon.items()}

    tp = fp = fn = 0
    for (_, toks, _), gold in zip(test_docs, gold_rows):
        gold_set = set(spans_of(gold))
        found = set()
        i = 0
        while i < len(toks):
            hit = None
            for length in range(min(6, len(toks) - i), 0, -1):   # найдовший збіг першим
                candidate = ' '.join(toks[i:i + length])
                if candidate in lexicon:
                    hit = (i, i + length, lexicon[candidate])
                    break
            if hit:
                found.add(hit)
                i = hit[1]
            else:
                i += 1
        tp += len(gold_set & found)
        fp += len(found - gold_set)
        fn += len(gold_set - found)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0), len(lexicon)


memory_f1, memory_p, memory_r = [], [], []
print('%-8s %10s %10s %10s' % ('зерно', 'модель', 'словник', 'різниця'))
for r in results:
    p, rec, f1, size = memory_lookup(r['train'], r['test'], r['gold'])
    memory_f1.append(f1); memory_p.append(p); memory_r.append(rec)
    print('%-8d %10.4f %10.4f %+10.4f   (словник %d форм)'
          % (r['seed'], r['f1'], f1, f1 - r['f1'], size))

print()
print('модель  :', pile([r['f1'] for r in results]))
print('словник :', pile(memory_f1))
print('  precision словника:', pile(memory_p))
print('  повнота словника  :', pile(memory_r))

## 14 · Перевірка власного числа: чи це висновок про NER — чи про наш код?

Якщо в тебе, як і в нас, словник виграв у моделі, — це надто гучний результат,
щоб лишити його без перевірки. Він цілком може бути твердженням не про NER, а
про **нашу розмітку**. Ставимо три контролі.

**Контроль перший: чи це не спосіб зіставляти?** Замінюємо найдовший збіг на
найпримітивніше можливе: позначати лише **однослівні** збіги.

**Контроль другий: чи мітка взагалі залежить від контексту?** Для кожної
однослівної форми, яка десь є сутністю, рахуємо, у якій частці своїх уживань
у корпусі вона позначена сутністю. Якщо ця частка близька до одиниці — контекст
у мітках не закодований, і модель фізично не може навчитися з них нічого
понад словник.

**Контроль третій: чи повнота словника дорівнює частці бачених форм?** Якщо
так, це підпис розмітки, зробленої тим самим зіставленням: словник ніколи не
проґавлює форму, яку знає.

In [ ]:
# --- контроль 1: найпримітивніше зіставлення
def memory_single_token(train_docs, test_docs, gold_rows):
    lexicon = defaultdict(Counter)
    for _, toks, labs in train_docs:
        for a, b, kind in spans_of(labs):
            lexicon[' '.join(toks[a:b])][kind] += 1
    lexicon = {t: c.most_common(1)[0][0] for t, c in lexicon.items()}
    tp = fp = fn = 0
    for (_, toks, _), gold in zip(test_docs, gold_rows):
        gold_set = set(spans_of(gold))
        found = {(i, i + 1, lexicon[w]) for i, w in enumerate(toks) if w in lexicon}
        tp += len(gold_set & found); fp += len(found - gold_set); fn += len(gold_set - found)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0


simple_f1 = [memory_single_token(r['train'], r['test'], r['gold']) for r in results]
print('КОНТРОЛЬ 1 — найпримітивніше зіставлення (лише однослівні збіги)')
print('   словник :', pile(simple_f1))
print('   модель  :', pile([r['f1'] for r in results]))
print()

# --- контроль 2: чи залежить мітка від контексту
inside = Counter(); outside = Counter()
for _, toks, labs in documents:
    covered = set()
    for a, b, _ in spans_of(labs):
        if b - a == 1:
            inside[toks[a]] += 1
        covered.update(range(a, b))
    for i, w in enumerate(toks):
        if i not in covered:
            outside[w] += 1

forms = set(inside)
uses_total = sum(inside[w] + outside[w] for w in forms)
uses_inside = sum(inside[w] for w in forms)
types_of_form = defaultdict(set)
for _, toks, labs in documents:
    for a, b, kind in spans_of(labs):
        if b - a == 1:
            types_of_form[toks[a]].add(kind)
two_types = [w for w, kinds in types_of_form.items() if len(kinds) > 1]

print('КОНТРОЛЬ 2 — чи визначається мітка самим рядком')
print('   однослівних форм-сутностей :', len(forms))
print('   усіх їхніх уживань         :', uses_total)
print('   позначено сутністю         : %d (%.4f)' % (uses_inside, uses_inside / uses_total))
print('   форм із ДВОМА типами       :', len(two_types))
print()

# --- контроль 3: повнота словника проти частки бачених форм
print('КОНТРОЛЬ 3 — повнота словника проти частки бачених форм')
for i, r in enumerate(results):
    print('   зерно %d: повнота %.4f · частка бачених %.4f · збігається: %s'
          % (r['seed'], memory_r[i], share_known[i],
             'так' if abs(memory_r[i] - share_known[i]) < 1e-9 else 'ні'))

Що з цього випливає — і це головний висновок зошита.

На розмітці, зробленій **проєкцією словника**, словник оптимальний **за
побудовою**. Мітка визначається самим рядком символів, контексту в ній майже
немає, і модель не може навчитися з таких міток нічого понад той самий
словник. Тому наш стенд **не здатний** відповісти на питання «чи вміє модель
читати контекст» — на це потрібен корпус, розмічений людиною, яка читала саме
цей текст.

А от на що стенд відповісти **здатний** — і відповідь від розмітки не
залежить: точність по токенах на такій задачі майже насичена ще до того, як
зʼявилась модель. Це властивість **метрики** й розподілу міток, а не того,
хто ці мітки поставив.

Це приклад звички, яку варто винести звідси у всю подальшу роботу:
**коли твій результат надто гучний, спитай спершу, чи це не властивість твого
коду**.

## 15 · Тип сутності важить більше за модель

Загальне F1 складається з трьох дуже різних чисел. Подивімось окремо по типах.

In [ ]:
print(classification_report(results[0]['gold'], results[0]['pred'], digits=4))

per_type = defaultdict(list)
for r in results:
    report = classification_report(r['gold'], r['pred'], output_dict=True, zero_division=0)
    for kind, values in report.items():
        if kind not in ('micro avg', 'macro avg', 'weighted avg'):
            per_type[kind].append(values['f1-score'])

print('по трьох зернах:')
for kind in sorted(per_type):
    print('   %-5s %s' % (kind, pile(per_type[kind])))
print()
print('LIC — закритий список: ідентифікатори ліцензій беруться зі стандартного')
print('      переліку й мають упізнавану форму, тому вивчається візерунок');
print('PROD — відкритий список: назви програм вигадують люди без правил,')
print('      і вивчити там можна лише самі рядки')

## 16 · Українська: чому все попереднє тут не працює

Усе, що ми міряли, — англійський текст. Для української розміченого корпусу
NER офлайн узяти нізвідки, тому далі буде **демонстрація, а не замір**: ми
покажемо, що саме ламається, на даних, які є, і не робитимемо з цього оцінок
якості.

Довідник, який є на кожній машині, — українські переклади міжнародних кодів
`iso-codes`: назви країн, мов, валют, систем письма. Це готовий газетир, а
корпус українського тексту візьмемо там само, де його бере весь курс: у
перекладах повідомлень програм.

In [ ]:
def load_catalog(name):
    """Пари «англійський оригінал → український переклад» з одного .mo-файлу."""
    path = '/usr/share/locale/uk/LC_MESSAGES/%s.mo' % name
    with open(path, 'rb') as f:
        catalog = gettext.GNUTranslations(f)._catalog
    return {s: d for s, d in catalog.items()
            if isinstance(s, str) and isinstance(d, str) and s and d}


CATALOGS = [('країни', 'iso_3166-1'), ('мови', 'iso_639-2'),
            ('валюти', 'iso_4217'), ('системи письма', 'iso_15924')]

catalogs = {}
for title, filename in CATALOGS:
    try:
        catalogs[title] = load_catalog(filename)
    except (FileNotFoundError, OSError):
        pass

if 'країни' not in catalogs:
    print('Пакета iso-codes з українськими перекладами в системі немає,')
    print('тому українську частину зошита пропускаємо.')
    print('Щоб вона запрацювала, постав пакет iso-codes (Fedora) або')
    print('iso-codes (Debian) разом з українськими локалями.')
else:
    print('%-16s %6s' % ('довідник', 'назв'))
    for title in catalogs:
        print('%-16s %6d' % (title, len(catalogs[title])))
    print('%-16s %6d' % ('РАЗОМ', sum(len(c) for c in catalogs.values())))

### 16.1 · Велика літера перестає бути ознакою

У NER для англійської найсильніша окрема ознака — велика літера: власні назви
пишуться з великої, звичайні слова ні. Порахуймо, чи це так для української,
на тому самому змісті двома мовами.

In [ ]:
if 'країни' in catalogs:
    print('%-16s %6s %14s %14s' % ('довідник', 'назв', 'з великої англ', 'з великої укр'))
    for title, pairs in catalogs.items():
        share_en = sum(1 for s in pairs) and sum(1 for s in pairs if s[:1].isupper()) / len(pairs)
        share_uk = sum(1 for d in pairs.values() if d[:1].isupper()) / len(pairs)
        print('%-16s %6d %14.4f %14.4f' % (title, len(pairs), share_en, share_uk))
    print()
    print('⚠️ рядки «країни» й «валюти» довіряй не повністю: у довіднику кожна назва')
    print('   стоїть окремим записом списку, а записи прийнято починати з великої')
    print('   літери незалежно від правопису. «Алжирський динар» у реченні — з малої.')
    print('   А от рядки «мови» й «системи письма» саме мовні: там українська пише')
    print('   з малої навіть у форматі, який штовхає до великої.')

### 16.2 · Відмінювання: скільки форм в однієї назви

Морфологічний аналізатор `pymorphy3` уміє розгорнути слово в усі його форми.
Подивімось, скільки їх і чи мають вони спільний початок, за яким їх можна було б
ловити.

In [ ]:
try:
    import pymorphy3
    morph = pymorphy3.MorphAnalyzer(lang='uk')
except Exception as err:
    morph = None
    print('pymorphy3 з українськими словниками недоступний:', err)
    print('постав pymorphy3 і pymorphy3-dicts-uk, щоб побачити цю частину')

if morph is not None:
    def all_forms(word):
        parsed = morph.parse(word.lower())
        return sorted({f.word for f in parsed[0].lexeme if f.word}) if parsed else []

    def common_start(words):
        return os.path.commonprefix(sorted(words)) if words else ''

    print('%-12s %5s  %-10s %s' % ('назва', 'форм', 'спільний', 'форми'))
    for name in ['Німеччина', 'Франція', 'Польща', 'Литва', 'Україна', 'Київ']:
        forms = all_forms(name)
        print('%-12s %5d  %-10s %s' % (name, len(forms), common_start(forms),
                                       ', '.join(forms[:8])))
    print()
    print('«Київ» — окремий випадок і дуже український: при відмінюванні і в')
    print('закритому складі переходить у е («Київ», але «Києва»), тому спільний')
    print('початок усіх форм — дві літери, під які підпадає пів словника.')
    print('Те саме з Львовом, Харковом і будь-яким містом на -ів.')

### 16.3 · Точний пошук проти пошуку за лемою

Тепер головне. Візьмімо однослівні назви країн і пошукаймо їх у корпусі
українських повідомлень двома способами: точним збігом рядка й збігом за
початковою формою (лемою). Здається очевидним, що другий спосіб кращий.

In [ ]:
def load_messages(min_len=30):
    """Українські переклади повідомлень програм."""
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue
        for source, target in catalog.items():
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > min_len and 'Project-Id' not in target:
                out.append(target)
    return out


UK_TOKEN = re.compile(r"[А-ЯІЇЄҐа-яіїєґ']+")

if morph is not None and 'країни' in catalogs:
    messages = load_messages()
    country_names = sorted({d for d in catalogs['країни'].values() if ' ' not in d})

    lemma_to_name = {}
    for name in country_names:
        parsed = morph.parse(name.lower())
        if parsed:
            lemma_to_name.setdefault(parsed[0].normal_form, name)

    lemma_cache = {}
    def lemma_of(word):
        if word not in lemma_cache:
            parsed = morph.parse(word)
            lemma_cache[word] = parsed[0].normal_form if parsed else word
        return lemma_cache[word]

    exact_hits = Counter()
    lemma_hits = defaultdict(Counter)
    for text in messages:
        for word in UK_TOKEN.findall(text):
            if word in country_names:
                exact_hits[word] += 1
            name = lemma_to_name.get(lemma_of(word.lower()))
            if name:
                lemma_hits[name][word] += 1

    total_lemma = sum(sum(c.values()) for c in lemma_hits.values())
    print('повідомлень у корпусі        :', len(messages))
    print('однослівних назв країн       :', len(country_names))
    print('точний збіг  : %4d уживань, %3d різних назв'
          % (sum(exact_hits.values()), len(exact_hits)))
    print('збіг за лемою: %4d уживань, %3d різних назв' % (total_lemma, len(lemma_hits)))
    print()
    print('%-14s %7s %7s   найчастіші словоформи' % ('назва', 'точно', 'лема'))
    for name, forms in sorted(lemma_hits.items(), key=lambda kv: -sum(kv[1].values()))[:10]:
        print('%-14s %7d %7d   %s'
              % (name, exact_hits.get(name, 0), sum(forms.values()),
                 ', '.join('%s x%d' % (w, n) for w, n in forms.most_common(3))))

Дивимось на верх таблиці. Якщо в тебе, як і в нас, там опинились **Того**,
**Малі** й **Перу** з сотнями «згадок» — то це не згадки країн. Це займенник
«того», прикметник «малі» й іменник «перо»: після зведення до леми велика
літера зникає разом із відмінком, і остання підказка зникає теж.

Порахуймо ціну цього обміну прямо.

In [ ]:
HOMONYMS = {'Того', 'Малі', 'Перу', 'Чад', 'Куба', 'Гана', 'Ямайка'}

if morph is not None and 'країни' in catalogs:
    bad_lemma = sum(sum(lemma_hits[n].values()) for n in HOMONYMS if n in lemma_hits)
    bad_exact = sum(exact_hits.get(n, 0) for n in HOMONYMS)
    total_exact = sum(exact_hits.values())

    print('назви-омоніми звичайних українських слів:',
          ', '.join(sorted(n for n in HOMONYMS if n in lemma_hits)))
    print()
    print('%-16s %9s %9s' % ('', 'точно', 'за лемою'))
    print('%-16s %9d %9d' % ('знайдено', total_exact, total_lemma))
    print('%-16s %9d %9d' % ('з них омоніми', bad_exact, bad_lemma))
    print('%-16s %9d %9d' % ('лишається чистих', total_exact - bad_exact, total_lemma - bad_lemma))
    print('%-16s %9.4f %9.4f' % ('точність',
                                 (total_exact - bad_exact) / total_exact,
                                 (total_lemma - bad_lemma) / total_lemma))
    print()
    print('справжніх знахідок додалось :',
          (total_lemma - bad_lemma) - (total_exact - bad_exact))
    print('хибних знахідок додалось    :', bad_lemma - bad_exact)

Це і є український урок теми. **Наївне лікування відмінювання не розвʼязує
задачу, а обмінює одну помилку на іншу**: трохи більше повноти за велику
втрату точності. Причина не в лематизаторі — він працює правильно. Причина в
тому, що українські назви країн є омонімами звичайних слів, щойно з них зняти
відмінок.

Висновок, який переживе цей зошит: для української **пошук за списком не є
рубежем навіть у першому наближенні**. Тут потрібен контекст — тобто саме те,
з чого починалась уся тема.

## 17 · Скільки все це коштувало

In [ ]:
spent = time.process_time() - STARTED
print('процесорного часу зошита: %d с (%.1f хв)' % (spent, spent / 60))
print('навантаження машини зараз:', round(os.getloadavg()[0], 2))
print()
print('⚠️ стінного часу пройшло більше — на завантаженій машині значно більше.')
print('   Процесорний час не залежить від сусідів по машині, тому цитуємо його.')

## Що спробувати самостійно

### 🟢 Рівень 1 — прогнати все на старій розмітці

У розділі 5 ми побачили, що старе правило `first_part` давало на третину
більше сутностей — і що більшість із них були сміттям. Але наскільки
змінюються від цього **метрики**?

Побудуй `documents` на старому правилі (`build_gazetteer(packages,
org_rule=first_part)`) і повтори розділи 8-13.

**Зроблено, якщо:** є таблиця з чотирьох рядків — рубіж «усе `O`», точність
по токенах, F1 по сутностях і рубіж «словник», — для обох розміток, і ти
можеш словами сказати, яке з чотирьох чисел змінилось найсильніше й чому.
Підказка, куди дивитись: сміття було переважно **дуже частими** словами, а
частий рядок легко вивчити напамʼять.

### 🟡 Рівень 2 — оціни, скільки шуму лишилось

Ми знайшли чотири підозрілі рядки в новому списку `ORG` за правилом «мало
джерел, багато міток». Правило грубе: воно проґавить слово, яке потрапило в
список від трьох різних пакетів, і навпаки — зачепить справжню організацію,
про яку в базі є один пакет.

Придумай і застосуй **другу** ознаку підозрілості, незалежну від першої.
Наприклад: чи трапляється цей рядок у корпусі як звичайне слово з малої
літери частіше, ніж з великої.

**Зроблено, якщо:** названо другу ознаку, показано список рядків, які вона
знаходить, і сказано, скільки з них **не** знайшла перша ознака. Одне речення
про те, чи можна взагалі скласти повний список хибних рядків, обовʼязкове.

### 🔴 Рівень 3 — додати посимвольну ознаку

Головна біда розділу 12 — небачені форми. Спробуй її зменшити, не міняючи
моделі: додай до ознак **посимвольні n-грами самого слова** (усі підрядки
довжиною 2-4 літери, наприклад `lib`, `ibp`, `bpn`…). Ідея в тому, щоб
небачена назва `libfoobar` потрапляла близько до бачених `libpng` і `libxml`
через спільні шматки написання.

**Зроблено, якщо:** ти заміряв F1 на **небачених** формах до й після, на трьох
тих самих зернах, і сказав, чи різниця більша за розкид — і якщо ні, то так і
написав.